
# Layer 1 — Dynamic Factor Model Backbone

This notebook implements **Layer 1** of the project:

> **Real-Time GDP Growth Nowcasting using a Hybrid Dynamic Factor Model with Machine Learning Residual Correction**

Scope in this notebook:

- audit the **actual local repository** before modeling;
- preserve the repository’s **first-day-of-month / first-day-of-quarter** timestamp semantics;
- construct a **vintage-aware GDP target history** and separate **truth tables**;
- ingest and transform the **monthly FRED-MD vintage files**;
- align monthly predictors with quarterly GDP in a **mixed-frequency state-space DFM**;
- estimate the DFM via **EM + Kalman filter / smoother**;
- produce **GDP nowcasts**, **news decomposition**, **coverage diagnostics**, and **Layer 2 artifacts**.

This notebook is aligned to **Section 3** of the methodological outline and the final project write-up, and uses the stable-subset block/tcode design from the uploaded data dictionary. The Layer 2 residual-correction model is **not** estimated here; the notebook exports the objects it will need.



## Non-negotiable date rule

The repository does **not** use artificial month-end timestamps as its native convention.

This notebook therefore:

- treats strings such as `2026-01-01` as **January 2026**, not as a month-end date;
- treats quarter markers using **period semantics** and sequence-aware parsing;
- avoids invalid synthetic dates such as `2026-02-29`;
- only converts to timestamp form when a downstream library requires it, and then uses the **period start timestamp** rather than a fabricated month-end.

Internally, audit and alignment logic use `pandas.Period` objects wherever possible.


In [ ]:
from __future__ import annotations

import importlib
import json
import math
import hashlib
import platform
import sys
import warnings
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

try:
    import dfm_layer1_utils as _layer1_utils
    _layer1_utils = importlib.reload(_layer1_utils)

    from dfm_layer1_utils import (
        ProtocolConfig,
        apply_tcodes_to_snapshot,
        as_model_index,
        build_factor_mapping,
        build_quarterly_target_series_for_vintage,
        build_repo_catalog,
        build_target_and_truth_objects,
        choose_canonical_md_manifest,
        choose_canonical_qd_manifest,
        choose_vintage_schedule,
        completion_checklist_frame,
        compute_block_coverage,
        extract_nowcast_from_results,
        fit_dfm_single_vintage,
        get_quarter_end_month,
        infer_period_frequency_from_values,
        inspect_csv_schema,
        inspect_excel_workbook,
        load_fred_snapshot,
        locate_repo_root,
        make_diagnostics_row,
        model_prediction_index_audit,
        oriented_factor_states,
        parse_periodish,
        prepare_mixed_frequency_model_inputs,
        protocol_summary_frame,
        run_layer1_dfm,
        select_monthly_panel,
        select_target_workbooks,
        serialize_protocol,
        stable_subset_metadata,
        summarize_manifest,
        within_quarter_origin,
        quarter_of_vintage,
    )
except ImportError as exc:
    raise ImportError(
        "dfm_layer1_utils.py was not found next to the notebook. "
        "Place the helper file in the repository root (or the notebook working directory) and rerun."
    ) from exc

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:

import statsmodels
import openpyxl

env = pd.DataFrame(
    [
        {"component": "python", "version": platform.python_version()},
        {"component": "pandas", "version": pd.__version__},
        {"component": "numpy", "version": np.__version__},
        {"component": "statsmodels", "version": statsmodels.__version__},
        {"component": "openpyxl", "version": openpyxl.__version__},
    ]
)
display(env)


## 1. Project setup

In [ ]:

# If the notebook is placed inside the repository (recommended), auto-detection should work.
# Otherwise, set REPO_ROOT_HINT to the absolute local path of the cloned repository.
REPO_ROOT_HINT = None

REPO_ROOT = Path(REPO_ROOT_HINT) if REPO_ROOT_HINT else locate_repo_root(Path.cwd())
if REPO_ROOT is None or not REPO_ROOT.exists():
    raise FileNotFoundError(
        "Could not locate the Nowcasting-GDP-Growth repository automatically. "
        "Set REPO_ROOT_HINT to the local repository path and rerun."
    )

RAW_DIR = REPO_ROOT / "data" / "raw"
OUTPUT_DIR = REPO_ROOT / "outputs" / "layer1_dfm"

print("Repository root:", REPO_ROOT)
print("Raw data directory:", RAW_DIR)
print("Output directory:", OUTPUT_DIR)


## 2. Repository audit and file mapping

In [ ]:

catalog = build_repo_catalog(REPO_ROOT)

catalog_summary = (
    catalog.groupby("group", dropna=False)
    .agg(
        n_files=("path", "size"),
        first_vintage=("vintage_period", "min"),
        last_vintage=("vintage_period", "max"),
        total_size_mb=("size_mb", "sum"),
    )
    .reset_index()
    .sort_values(["group"])
)

md_manifest = choose_canonical_md_manifest(catalog)
qd_manifest = choose_canonical_qd_manifest(catalog)
target_workbooks = select_target_workbooks(REPO_ROOT)

manifest_summary = pd.concat(
    [
        summarize_manifest(md_manifest, "Canonical monthly predictor manifest"),
        summarize_manifest(qd_manifest, "Canonical quarterly predictor manifest"),
    ],
    ignore_index=True,
)

workbook_map = pd.DataFrame(
    [
        {"object_name": k, "path": str(v.relative_to(REPO_ROOT))}
        for k, v in sorted(target_workbooks.items())
    ]
)

display(catalog_summary)
display(manifest_summary)
display(workbook_map)


In [ ]:

def hash_file(path: Path) -> str:
    return hashlib.md5(path.read_bytes()).hexdigest()

hash_checks = []

md_current = REPO_ROOT / "data" / "raw" / "FRED-MD-MONTHLY" / "current.csv"
if md_current.exists() and not md_manifest.empty:
    latest_md = REPO_ROOT / md_manifest.iloc[-1]["path"]
    hash_checks.append(
        {
            "file_group": "monthly current vs latest dated file",
            "current_path": str(md_current.relative_to(REPO_ROOT)),
            "latest_dated_path": str(latest_md.relative_to(REPO_ROOT)),
            "same_bytes": hash_file(md_current) == hash_file(latest_md),
        }
    )

qd_current = REPO_ROOT / "data" / "raw" / "FRED-QD-QUARTERLY" / "current.csv"
if qd_current.exists() and not qd_manifest.empty:
    latest_qd = REPO_ROOT / qd_manifest.iloc[-1]["path"]
    hash_checks.append(
        {
            "file_group": "quarterly current vs latest dated file",
            "current_path": str(qd_current.relative_to(REPO_ROOT)),
            "latest_dated_path": str(latest_qd.relative_to(REPO_ROOT)),
            "same_bytes": hash_file(qd_current) == hash_file(latest_qd),
        }
    )

display(pd.DataFrame(hash_checks))


## 3. Raw file inspection and schema understanding

In [ ]:

def compact_csv_inspection(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    info = inspect_csv_schema(path)
    summary = pd.DataFrame(
        [
            {
                "path": str(Path(info["path"]).relative_to(REPO_ROOT)),
                "date_col": info["date_col"],
                "inferred_freq": info["inferred_freq"],
                "metadata_row_count": info["metadata_row_count"],
                "has_tcode_row": info["has_tcode_row"],
                "n_rows": info["n_rows"],
                "n_cols": info["n_cols"],
                "min_timestamp": info["min_timestamp"],
                "max_timestamp": info["max_timestamp"],
            }
        ]
    )
    return summary, info["preview"]

representative_paths = []
if len(md_manifest) > 0:
    representative_paths.extend(
        [
            REPO_ROOT / md_manifest.iloc[0]["path"],
            REPO_ROOT / md_manifest.iloc[len(md_manifest) // 2]["path"],
            REPO_ROOT / md_manifest.iloc[-1]["path"],
        ]
    )
if len(qd_manifest) > 0:
    representative_paths.extend(
        [
            REPO_ROOT / qd_manifest.iloc[0]["path"],
            REPO_ROOT / qd_manifest.iloc[-1]["path"],
        ]
    )

sample_dir = REPO_ROOT / "data" / "sample"
if sample_dir.exists():
    sample_csvs = sorted(sample_dir.glob("*.csv"))
    if sample_csvs:
        representative_paths.append(sample_csvs[0])

for path in representative_paths:
    print(f"\nInspecting: {path.relative_to(REPO_ROOT)}")
    summary, preview = compact_csv_inspection(path)
    display(summary)
    display(preview)


In [ ]:

for workbook_name, workbook_path in sorted(target_workbooks.items()):
    print(f"\nWorkbook audit: {workbook_name}")
    display(inspect_excel_workbook(workbook_path))


## 4. Protocol locking and configuration


The protocol is locked **before** estimation:

- benchmark window starts at **2000Q1**;
- primary truth is **third-release GDP growth**;
- robustness truths are **latest RTDSM** and **GDPplus**;
- the main Layer 1 predictor flow is **monthly FRED-MD vintages**;
- the default shipped model uses the **stable subset** because the uploaded data dictionary provides an explicit ex ante block/tcode/anchor map for it;
- the DFM uses **1 global factor + 1 factor per block** and a low-order factor VAR;
- ragged edges stay **inside** the state-space system;
- Layer 2 is fed only with **vintage-legal** DFM outputs.


In [ ]:

RUN_MODE = "research"   # set to "debug" for a short smoke test; for a fast late-sample news() check, temporarily start at 2025Q4 with vintage_limit=5

if RUN_MODE == "debug":
    config = ProtocolConfig(
        repo_root=str(REPO_ROOT),
        output_dir=str(OUTPUT_DIR),
        benchmark_start_quarter="2000Q1",
        panel_mode="stable",
        truth_main="third_release",
        truth_robustness=("latest_rtdsm", "gdpplus"),
        candidate_factor_orders=(1, 2),
        fixed_factor_order=1,
        select_factor_order_per_vintage=False,
        idiosyncratic_ar1=True,
        em_maxiter=25,
        em_tolerance=1e-5,
        vintage_limit=6,
        min_monthly_obs=24,
        force_refit=True,
    )
else:
    config = ProtocolConfig(
        repo_root=str(REPO_ROOT),
        output_dir=str(OUTPUT_DIR),
        benchmark_start_quarter="2000Q1",
        panel_mode="stable",
        truth_main="third_release",
        truth_robustness=("latest_rtdsm", "gdpplus"),
        candidate_factor_orders=(1, 2),
        fixed_factor_order=1,
        select_factor_order_per_vintage=False,
        idiosyncratic_ar1=True,
        em_maxiter=100,
        em_tolerance=1e-6,
        vintage_limit=None,
        min_monthly_obs=24,
        force_refit=True,
    )

display(protocol_summary_frame(config))
serialize_protocol(config, OUTPUT_DIR / "layer1_protocol.json")


## 5. Date convention audit and parsing rules


The local repository audit should verify the date semantics **from the files themselves** rather than from assumptions.

Observed conventions that this notebook is designed to support:

- **monthly FRED-MD snapshots** use a first-of-month date column (for example `1/1/1959`, `2/1/1959`, ...), typically with a leading `Transform:` row;
- **quarterly FRED-QD snapshots** use a quarter marker sequence such as `3/1/1959`, `6/1/1959`, ... together with leading `factors` and `transform` metadata rows;
- workbook-based target files may use quarter labels, quarter-start markers, or a vintage-by-observation matrix layout, so the notebook inspects them locally at runtime.

The parser therefore infers frequency from the **sequence** and metadata rows, not from a blind month-end conversion.


In [ ]:

def preview_date_parsing(path: Path, n: int = 8) -> pd.DataFrame:
    raw = pd.read_csv(path, usecols=[0], nrows=max(12, n))
    info = inspect_csv_schema(path)
    date_col = info["date_col"]
    skip = int(info["metadata_row_count"])
    raw_values = raw[date_col].iloc[skip: skip + n].astype(str).tolist()
    freq = info["inferred_freq"]
    periods = [parse_periodish(x, freq_hint=freq) for x in raw_values]
    timestamps = [p.to_timestamp() if (p is not None and not pd.isna(p)) else pd.NaT for p in periods]
    return pd.DataFrame(
        {
            "raw_date_string": raw_values,
            "inferred_freq": [freq] * len(raw_values),
            "parsed_period": [str(p) if p is not None and not pd.isna(p) else None for p in periods],
            "period_start_timestamp": timestamps,
        }
    )

latest_md_path = REPO_ROOT / md_manifest.iloc[-1]["path"]
display(preview_date_parsing(latest_md_path))

if len(qd_manifest) > 0:
    latest_qd_path = REPO_ROOT / qd_manifest.iloc[-1]["path"]
    display(preview_date_parsing(latest_qd_path))

assert inspect_csv_schema(latest_md_path)["max_timestamp"].day == 1, "Monthly repo timestamps should remain first-of-month markers."


## 6. Vintage-aware target construction

In [ ]:

target_objects = build_target_and_truth_objects(REPO_ROOT)

target_summary_rows = []
for name, obj in target_objects.items():
    if isinstance(obj, pd.DataFrame):
        summary_row = {
            "object_name": name,
            "n_rows": len(obj),
            "columns": ", ".join(map(str, obj.columns.tolist()[:12])),
        }
        for candidate in ["vintage_period", "obs_period", "quarter"]:
            if candidate in obj.columns:
                summary_row[f"{candidate}_min"] = obj[candidate].min()
                summary_row[f"{candidate}_max"] = obj[candidate].max()
        target_summary_rows.append(summary_row)

target_summary = pd.DataFrame(target_summary_rows)
display(target_summary)

target_vintage_table = target_objects["target_vintage_table"]
display(target_vintage_table.head(10))

if "truth_release_table" in target_objects:
    display(target_objects["truth_release_table"].head(10))
if "truth_third_release" in target_objects:
    display(target_objects["truth_third_release"].head(10))
if "truth_latest" in target_objects:
    display(target_objects["truth_latest"].head(10))
if "truth_gdpplus" in target_objects:
    display(target_objects["truth_gdpplus"].head(10))


## 7. Monthly data ingestion from actual CSV files

In [ ]:

vintage_schedule = choose_vintage_schedule(
    md_manifest,
    start_quarter=config.benchmark_start_quarter,
    vintage_limit=config.vintage_limit,
)

if len(vintage_schedule) == 0:
    raise ValueError("No eligible month-end vintages found for the configured benchmark window.")

example_vintage = vintage_schedule[min(len(vintage_schedule) - 1, 5 if len(vintage_schedule) > 5 else 0)]
example_md_row = md_manifest.loc[md_manifest["vintage_period"] == example_vintage].iloc[0]
example_md_path = REPO_ROOT / example_md_row["path"]

example_snapshot = load_fred_snapshot(example_md_path, freq_hint="M")
example_transformed = apply_tcodes_to_snapshot(example_snapshot)
example_panel, example_meta = select_monthly_panel(example_transformed, panel_mode=config.panel_mode)

stable_meta = stable_subset_metadata()
stable_presence = (
    stable_meta.assign(in_example_snapshot=stable_meta["mnemonic"].isin(example_snapshot["data"].columns))
    .groupby(["block_label"], as_index=False)
    .agg(n_present=("in_example_snapshot", "sum"), n_stable_series=("in_example_snapshot", "size"))
)

print("Example monthly vintage:", example_vintage)
print("Example monthly file:", example_md_path.relative_to(REPO_ROOT))
display(stable_presence)
display(example_meta.head(20))
display(example_panel.tail(8))


## 8. Transformation and standardization pipeline


Pipeline order in this implementation:

1. freeze the monthly vintage snapshot;
2. apply the **official FRED transformation code** from the file or stable-subset metadata;
3. keep the ragged edge as missing values;
4. let the DFM standardize **within the vintage-specific estimation sample** during model fitting.

The notebook does **not** standardize once on the full sample and reuse those moments across vintages.


In [ ]:

example_series = example_panel.columns[0]
example_tcode = (
    int(example_snapshot["tcodes"].get(example_series))
    if example_series in example_snapshot["tcodes"].index
    else int(example_meta.set_index("mnemonic").loc[example_series, "tcode"])
)

transformation_audit = pd.DataFrame(
    {
        "raw_value": example_snapshot["data"][example_series].tail(8).values,
        "transformed_value": example_transformed[example_series].tail(8).values,
    },
    index=example_transformed.index[-8:].astype(str),
)
transformation_audit.index.name = "period"

print("Example series:", example_series)
print("Applied tcode:", example_tcode)
display(transformation_audit)


## 9. Mixed-frequency monthly–quarterly alignment


The mixed-frequency bridge is built as follows:

- monthly indicators stay on a monthly `PeriodIndex`;
- GDP history is reconstructed vintage-by-vintage on a quarterly `PeriodIndex`;
- quarterly GDP enters the state-space system only as a quarterly series;
- internally, `statsmodels.DynamicFactorMQ` receives first-day timestamps generated from the monthly/quarterly periods, but the underlying semantics remain period-based.

The methodological bridge follows the Mariano–Murasawa-style quarterly aggregation logic described in Section 3.


In [ ]:

example_target_quarter = quarter_of_vintage(example_vintage)
example_quarterly_target_full = build_quarterly_target_series_for_vintage(target_vintage_table, example_vintage)
example_panel_model, example_quarterly_target = prepare_mixed_frequency_model_inputs(
    example_panel,
    example_quarterly_target_full,
    quarterly_target_name="gdp_growth",
)

alignment_demo = pd.DataFrame(
    {
        "monthly_period": example_panel.index[-6:].astype(str),
        "monthly_timestamp_start": example_panel.index[-6:].to_timestamp(),
        "mapped_quarter": example_panel.index[-6:].to_timestamp().to_period("Q").astype(str),
    }
)

alignment_span = pd.DataFrame(
    [
        {
            "monthly_start": example_panel_model.index.min(),
            "monthly_end": example_panel_model.index.max(),
            "raw_quarterly_start": example_quarterly_target_full.index.min(),
            "raw_quarterly_end": example_quarterly_target_full.index.max(),
            "aligned_quarterly_start": example_quarterly_target.index.min(),
            "aligned_quarterly_end": example_quarterly_target.index.max(),
            "raw_quarters": len(example_quarterly_target_full),
            "aligned_quarters": len(example_quarterly_target),
        }
    ]
)

print("Example vintage:", example_vintage)
print("Target quarter for this vintage:", example_target_quarter)
print("Quarter-end month marker:", get_quarter_end_month(example_target_quarter))
display(alignment_demo)
display(alignment_span)
display(example_quarterly_target.tail(8))


## 10. DFM specification in state-space form


For monthly indicator \(i\) in block \(b(i)\), the measurement system is implemented as:

\[
x^{(v)}_{i,m} = \mu_i + \lambda^{(g)}_i f_{g,m} + \lambda_{i,b(i)} f_{b(i),m} + u_{i,m},
\]

with idiosyncratic AR(1) dynamics

\[
u_{i,m} = \rho_i u_{i,m-1} + \varepsilon_{i,m},
\]

and factor dynamics

\[
F_m = A_1 F_{m-1} + \cdots + A_p F_{m-p} + \eta_m.
\]

Implementation choice:

- **1 global factor**;
- **1 factor per economic block**;
- low-order factor VAR, with \(p\) fixed here to 1 by default (the notebook also exposes a small candidate grid if you want to select it within-sample).

The mixed-frequency monthly/quarterly structure, Kalman filter/smoother, EM estimation, and news decomposition are handled by `statsmodels.tsa.statespace.dynamic_factor_mq.DynamicFactorMQ`.


In [ ]:

factor_map, factor_orders = build_factor_mapping(
    example_panel.columns,
    example_meta,
    quarterly_target_name="gdp_growth",
)
factor_orders = {k: int(config.fixed_factor_order or 1) for k in factor_orders}

factor_map_preview = pd.DataFrame(
    [{"series": k, "factors": ", ".join(v)} for k, v in factor_map.items()]
).head(20)

display(factor_map_preview)
display(pd.DataFrame([{"factor_group": str(k), "var_order_p": v} for k, v in factor_orders.items()]))


## 11. Estimation via EM + Kalman Filter / Kalman Smoother

In [ ]:

example_model, example_results = fit_dfm_single_vintage(
    monthly_panel=example_panel,
    quarterly_target=example_quarterly_target,
    panel_meta=example_meta,
    factor_order=int(config.fixed_factor_order or 1),
    idiosyncratic_ar1=config.idiosyncratic_ar1,
    em_maxiter=config.em_maxiter if RUN_MODE == "debug" else min(config.em_maxiter, 50),
    em_tolerance=config.em_tolerance,
    quarterly_target_name="gdp_growth",
)

example_diag = make_diagnostics_row(
    vintage_period=example_vintage,
    target_quarter=example_target_quarter,
    monthly_panel=example_panel,
    results=example_results,
    factor_order=int(config.fixed_factor_order or 1),
)

display(pd.DataFrame([example_diag]))



### Debug audit of the failing `news()` path

The failing path in the original implementation was:

`target_quarter -> impact_date -> prev_results.news(...)`

The critical issue was not the repository's first-of-period timestamp convention.  
The issue was that the **mixed-frequency model's internal prediction index** became unsupported inside statsmodels.

Why that happened:

- the monthly FRED-MD panel starts in **1959-01**;
- the real-time GDP vintage table starts in **1947Q1**;
- `DynamicFactorMQ.construct_endog(...)` concatenates the monthly index first and then appends quarterly-only months;
- if the quarterly history begins before the monthly panel, the internal monthly index becomes **non-monotonic**;
- statsmodels then **ignores** the date index and replaces it with a generated integer index;
- `news()` later calls `_get_prediction_index(...)` on the updated model, so an out-of-sample monthly impact date like `2026-03` can no longer be resolved there.

The helper patch therefore aligns the quarterly target history to the monthly support **before** fitting / applying `DynamicFactorMQ`, and validates the impact date on the **actual statsmodels model object** used by `news()`.


In [ ]:

example_impact_date = get_quarter_end_month(example_target_quarter)
example_index_audit = model_prediction_index_audit(example_results.model, example_impact_date)

display(example_index_audit)

assert not bool(example_index_audit.loc[0, "model_index_generated"]), "The example model should keep a supported date index."
assert bool(example_index_audit.loc[0, "model_index_monotonic"]), "The example model index should be monotonic increasing."
assert bool(example_index_audit.loc[0, "impact_date_supported"]), "The example impact date must be resolvable by the actual statsmodels model."


In [ ]:

llf_path = np.array(json.loads(example_diag["llf_path_json"]))
plt.figure(figsize=(8, 4))
plt.plot(llf_path, marker="o")
plt.title(f"EM log-likelihood path — example vintage {example_vintage}")
plt.xlabel("EM iteration")
plt.ylabel("log-likelihood")
plt.grid(True, alpha=0.3)
plt.show()

example_state_tail = oriented_factor_states(example_results, example_meta, kind="smoothed").tail(12)
display(example_state_tail)

standardization_audit = pd.DataFrame(
    {
        "series": list(example_results.model._endog_mean.index),
        "mean_used_by_model": list(example_results.model._endog_mean.values),
        "std_used_by_model": list(example_results.model._endog_std.values),
    }
).head(20)
display(standardization_audit)


## 12. GDP nowcast generation

In [ ]:

example_nowcast = extract_nowcast_from_results(
    results=example_results,
    vintage_period=example_vintage,
    target_quarter=example_target_quarter,
    quarterly_target_name="gdp_growth",
)
print("Example DFM nowcast:", example_nowcast)



The next cell runs the full pseudo-real-time Layer 1 loop:

1. freeze monthly vintage \(v\);
2. rebuild the GDP history visible at \(v\);
3. transform the predictor panel at \(v\);
4. estimate the DFM by EM + Kalman;
5. compute the nowcast for the current quarter;
6. compute news relative to the previous vintage;
7. export Layer 2 artifacts.


In [ ]:

outputs = run_layer1_dfm(config)

nowcasts_df = outputs["nowcasts"]
states_df = outputs["states"]
news_series_df = outputs["news_series"]
news_blocks_df = outputs["news_blocks"]
coverage_df = outputs["coverage"]
diagnostics_df = outputs["diagnostics"]

display(nowcasts_df.head(10))
display(nowcasts_df.tail(10))


In [ ]:

run_validation = pd.DataFrame(
    [
        {
            "n_nowcast_rows": len(nowcasts_df),
            "n_state_rows": len(states_df),
            "n_news_series_rows": len(news_series_df),
            "n_news_block_rows": len(news_blocks_df),
            "last_vintage_period": nowcasts_df["vintage_period"].max() if len(nowcasts_df) else pd.NaT,
            "last_target_quarter": nowcasts_df["target_quarter"].max() if len(nowcasts_df) else pd.NaT,
            "any_generated_model_index": bool(diagnostics_df["model_index_generated"].fillna(False).any()) if "model_index_generated" in diagnostics_df.columns else np.nan,
        }
    ]
)
display(run_validation)

assert len(nowcasts_df) > 0, "run_layer1_dfm(config) should produce at least one nowcast row."
assert len(news_series_df) > 0, "The patched pipeline should now produce news-series rows."
assert len(news_blocks_df) > 0, "The patched pipeline should now produce block-level news rows."
if "model_index_generated" in diagnostics_df.columns:
    assert not diagnostics_df["model_index_generated"].fillna(False).any(), "No fitted vintage should fall back to a generated integer index."


## 13. News decomposition

In [ ]:

nonempty_news_blocks = news_blocks_df.copy()
if "signed_block_news" in nonempty_news_blocks.columns:
    nonempty_news_blocks = nonempty_news_blocks.dropna(subset=["signed_block_news"], how="all")

display(nonempty_news_blocks.tail(20))
display(news_series_df.tail(20))


In [ ]:

if not nonempty_news_blocks.empty:
    latest_news_vintage = nonempty_news_blocks["vintage_period"].dropna().max()
    latest_news = (
        nonempty_news_blocks[nonempty_news_blocks["vintage_period"] == latest_news_vintage]
        .sort_values("signed_block_news", ascending=False)
        .reset_index(drop=True)
    )
    display(latest_news)

    plt.figure(figsize=(9, 4))
    plt.bar(latest_news["block_label"].astype(str), latest_news["signed_block_news"])
    plt.xticks(rotation=45, ha="right")
    plt.title(f"Signed block news — vintage {latest_news_vintage}")
    plt.ylabel("GDP nowcast impact")
    plt.grid(True, axis="y", alpha=0.3)
    plt.show()
else:
    print("No non-empty news decomposition rows were produced in this run.")


## 14. Diagnostics and validation

In [ ]:

diagnostic_summary = pd.DataFrame(
    [
        {
            "n_vintages_estimated": len(diagnostics_df),
            "share_converged_flag_true": diagnostics_df["converged_flag"].mean() if len(diagnostics_df) else np.nan,
            "median_em_iterations": diagnostics_df["em_iterations"].median() if len(diagnostics_df) else np.nan,
            "max_sample_end_period": diagnostics_df["sample_end_period"].max() if len(diagnostics_df) else pd.NaT,
            "share_model_index_generated_true": diagnostics_df["model_index_generated"].mean() if "model_index_generated" in diagnostics_df.columns and len(diagnostics_df) else np.nan,
        }
    ]
)
display(diagnostic_summary)

if {"third_release", "dfm_nowcast"}.issubset(nowcasts_df.columns):
    eval_df = nowcasts_df.dropna(subset=["third_release", "dfm_nowcast"]).copy()
    if len(eval_df):
        overall_rmsfe = np.sqrt(np.mean((eval_df["third_release"] - eval_df["dfm_nowcast"]) ** 2))
        rmsfe_by_tau = (
            eval_df.groupby("within_quarter_origin")
            .apply(lambda x: np.sqrt(np.mean((x["third_release"] - x["dfm_nowcast"]) ** 2)))
            .reset_index(name="rmsfe")
        )
        display(pd.DataFrame([{"overall_rmsfe_vs_third_release": overall_rmsfe}]))
        display(rmsfe_by_tau)
    else:
        print("Third-release truth is parsed, but no scored nowcasts are available in this run yet.")
else:
    print("Third-release truth is not available or not yet parsed; skip RMSFE display.")


In [ ]:

if len(diagnostics_df):
    plt.figure(figsize=(8, 4))
    plt.hist(diagnostics_df["em_iterations"].dropna(), bins=min(20, max(5, diagnostics_df["em_iterations"].nunique())))
    plt.title("Distribution of EM iterations across vintages")
    plt.xlabel("EM iterations")
    plt.ylabel("Count")
    plt.grid(True, axis="y", alpha=0.3)
    plt.show()

if len(coverage_df) and "coverage" in coverage_df.columns:
    avg_coverage = (
        coverage_df.groupby(["within_quarter_origin", "block_label"], dropna=False)["coverage"]
        .mean()
        .reset_index()
        .sort_values(["within_quarter_origin", "block_label"])
    )
    display(avg_coverage.head(20))

    plt.figure(figsize=(9, 4))
    plt.bar(avg_coverage["block_label"].astype(str), avg_coverage["coverage"])
    plt.xticks(rotation=45, ha="right")
    plt.title("Average block coverage in the current-quarter window")
    plt.ylabel("Coverage share")
    plt.grid(True, axis="y", alpha=0.3)
    plt.show()


## 15. Artifact export for Layer 2

In [ ]:

artifact_check = completion_checklist_frame(OUTPUT_DIR)
display(artifact_check)

artifact_purpose = pd.DataFrame(
    [
        {"artifact": "layer1_protocol.json", "purpose": "Locked Layer 1 experimental protocol"},
        {"artifact": "dfm_nowcasts.csv", "purpose": "Vintage-level DFM nowcasts, truths, and residual lags"},
        {"artifact": "dfm_states.parquet|csv", "purpose": "Current and lagged oriented factor states"},
        {"artifact": "dfm_news_series.csv", "purpose": "Series-level news decomposition"},
        {"artifact": "dfm_news_blocks.csv", "purpose": "Block-level signed and absolute news"},
        {"artifact": "dfm_coverage.csv", "purpose": "Block coverage / ragged-edge summaries"},
        {"artifact": "dfm_diagnostics.csv", "purpose": "EM convergence and model diagnostics"},
        {"artifact": "vintage_manifest_monthly.csv", "purpose": "Canonical monthly vintage manifest"},
        {"artifact": "vintage_manifest_quarterly.csv", "purpose": "Canonical quarterly vintage manifest"},
        {"artifact": "repository_catalog.csv", "purpose": "Repository-wide audit table"},
    ]
)
display(artifact_purpose)


## 16. Final completion checklist

In [ ]:

task_checklist = pd.DataFrame(
    [
        {"task": "Repository tree audited and catalogued", "completed": True, "where_to_check": "repository_catalog.csv and Sections 2–3"},
        {"task": "Monthly and quarterly date conventions audited from actual files", "completed": True, "where_to_check": "Section 5 date audit tables"},
        {"task": "Vintage-aware target history constructed from RTDSM workbook", "completed": True, "where_to_check": "Section 6 and target_vintage_table"},
        {"task": "Main truth table separated from vintage target history", "completed": True, "where_to_check": "truth_third_release / truth_latest / truth_gdpplus"},
        {"task": "Monthly predictor vintages ingested from actual CSV files", "completed": True, "where_to_check": "Sections 2, 3, and 7"},
        {"task": "Official transformations applied before standardization", "completed": True, "where_to_check": "Section 8"},
        {"task": "Statsmodels internal prediction index audited on the actual model object", "completed": True, "where_to_check": "Example index audit and diagnostics_df model_index_* columns"},
        {"task": "Mixed-frequency DFM estimated in state-space form", "completed": True, "where_to_check": "Sections 10–12"},
        {"task": "News decomposition produced", "completed": True, "where_to_check": "Section 13 and dfm_news_*.csv"},
        {"task": "Diagnostics and validation tables produced", "completed": True, "where_to_check": "Section 14 and dfm_diagnostics.csv"},
        {"task": "Layer 2 artifacts exported", "completed": True, "where_to_check": "Section 15 and outputs/layer1_dfm"},
    ]
)
display(task_checklist)



## 17. How to run

### File placement

Place **both** files in the local repository root:

- `layer1_dfm_backbone.ipynb`
- `dfm_layer1_utils.py`

A typical structure is:

```text
Nowcasting-GDP-Growth/
├── data/
├── dfm_layer1_utils.py
└── layer1_dfm_backbone.ipynb
```

### Required packages

Install the core environment:

```bash
pip install pandas numpy matplotlib openpyxl statsmodels pyarrow
```

`pyarrow` is recommended for Parquet export of factor states. If it is unavailable, the helper will fall back to CSV for that artifact.

### Execution order

1. open the notebook from the repository root;
2. run the notebook top to bottom once in **debug** mode if you want a short smoke test;
3. switch `RUN_MODE` to `"research"` for the full recursive Layer 1 run;
4. inspect the exported artifacts in:

```text
outputs/layer1_dfm/
```

### What to inspect first after a full run

- `repository_catalog.csv`
- `vintage_manifest_monthly.csv`
- `dfm_nowcasts.csv`
- `dfm_news_blocks.csv`
- `dfm_diagnostics.csv`

### Notes on extending to the full panel

The shipped default is the **stable subset** because the uploaded data dictionary provides an explicit ex ante block map, tcode map, and anchor map for it. The helper exposes a `panel_mode="full"` path, but if you want a fully block-structured full-panel production spec, extend the series-to-block metadata in `dfm_layer1_utils.py` so every monthly series has an explicit block assignment.


### Replace-helper note

If you replace `dfm_layer1_utils.py` while the notebook kernel is already running, restart the kernel before re-executing. The import cell also reloads the helper module explicitly so the notebook picks up the corrected parser and `news()` index logic.
